# Task 3 — LSTM + XGBoost + LightGBM Ensemble
**Goal:** Predict monthly average `x2` (grid load) per device for May–Oct 2025  
**Metric:** MAE (lower = better)  
**Train:** Oct 2024 – Apr 2025 | **Val:** May–Jun 2025 | **Test:** Jul–Oct 2025

## 1. Setup

In [ ]:
!pip install lightgbm xgboost pandas numpy scikit-learn pyarrow torch -q

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor
import xgboost as xgb

FORECAST_MONTHS = [5, 6, 7, 8, 9, 10]
FORECAST_YEAR   = 2025
RANDOM_STATE    = 42
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Torch device: {DEVICE}')

## 2. Config — local or Colab

In [ ]:
# ── CHANGE THESE ──────────────────────────────────────────────────────────────
ENV = 'colab'   # 'local' or 'colab'

LOCAL_DATA_DIR = '../solution_task3/data'
COLAB_DATA_DIR = '/content/drive/MyDrive/task3_data'

SUBSAMPLE_FRAC = 0.2   # 0.2 = ~2GB in RAM; raise to 1.0 for full data
CHUNK_SIZE     = 500_000
# ──────────────────────────────────────────────────────────────────────────────

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = COLAB_DATA_DIR
else:
    DATA_DIR = LOCAL_DATA_DIR

print(f'ENV={ENV}  DATA_DIR={DATA_DIR}  SUBSAMPLE_FRAC={SUBSAMPLE_FRAC}')

## 3. Load data in chunks with subsampling

In [ ]:
def load_chunked(path, chunk_size, subsample_frac, seed=42):
    rng = np.random.default_rng(seed)
    chunks = []
    reader = pd.read_csv(path, parse_dates=['Timedate'], chunksize=chunk_size)
    for i, chunk in enumerate(reader):
        if subsample_frac < 1.0:
            mask = rng.random(len(chunk)) < subsample_frac
            chunk = chunk[mask]
        chunks.append(chunk)
        if (i + 1) % 10 == 0:
            print(f'  chunk {i+1}, rows so far: {sum(len(c) for c in chunks):,}')
    return pd.concat(chunks, ignore_index=True)


print(f'Loading data.csv (chunk={CHUNK_SIZE:,}, subsample={SUBSAMPLE_FRAC}) ...')
df = load_chunked(os.path.join(DATA_DIR, 'data.csv'), CHUNK_SIZE, SUBSAMPLE_FRAC)

devices = pd.read_csv(os.path.join(DATA_DIR, 'devices.csv'))
df = df.merge(devices, on='deviceId', how='left')

print(f'\nLoaded {len(df):,} rows | {df["deviceId"].nunique()} devices')
print(f'Date range: {df["Timedate"].min()} → {df["Timedate"].max()}')
print(f'RAM: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

## 4. EDA

In [ ]:
print('--- missing values (%) ---')
print((df.isnull().mean() * 100).round(2))

In [ ]:
df['year']  = df['Timedate'].dt.year
df['month'] = df['Timedate'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['x2'].hist(bins=60, ax=axes[0])
axes[0].set_title('x2 distribution')

monthly_mean = df.groupby(['year', 'month'])['x2'].mean().reset_index()
monthly_mean['date'] = pd.to_datetime(monthly_mean[['year', 'month']].assign(day=1))
monthly_mean.set_index('date')['x2'].plot(ax=axes[1], marker='o')
axes[1].set_title('Monthly mean x2 (fleet-wide)')
plt.tight_layout()
plt.show()

In [ ]:
temp_cols = [f't{i}' for i in range(1, 14) if f't{i}' in df.columns]
corr = df[temp_cols + ['x1', 'x2']].corr()['x2'].drop('x2').sort_values()
corr.plot(kind='barh', figsize=(8, 5), title='Feature correlation with x2')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 5. Feature Engineering & monthly aggregation

In [ ]:
def build_features(df):
    df = df.copy()
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    if 'deviceType' in df.columns:
        le = LabelEncoder()
        df['deviceType_enc'] = le.fit_transform(df['deviceType'].astype(str))
    return df


def aggregate_monthly(df):
    temp_cols = [f't{i}' for i in range(1, 14) if f't{i}' in df.columns]
    agg = {c: 'mean' for c in temp_cols}
    for col in ['x1', 'x2', 'month_sin', 'month_cos']:
        if col in df.columns:
            agg[col] = 'mean'
    for col in ['latitude', 'longitude', 'deviceType_enc', 'x3']:
        if col in df.columns:
            agg[col] = 'first'
    return df.groupby(['deviceId', 'year', 'month']).agg(agg).reset_index()


df      = build_features(df)
monthly = aggregate_monthly(df)
del df  # free RAM

# Lag features on monthly table
monthly = monthly.sort_values(['deviceId', 'year', 'month']).reset_index(drop=True)
for lag in [1, 2, 3]:
    monthly[f'x2_lag{lag}'] = monthly.groupby('deviceId')['x2'].shift(lag)
monthly['x2_roll3'] = monthly.groupby('deviceId')['x2'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)

print(f'Monthly rows: {len(monthly):,}')
monthly.head()

## 6. Train / Validation split

In [ ]:
FEATURE_COLS = [
    'month_sin', 'month_cos',
    *[f't{i}' for i in range(1, 14) if f't{i}' in monthly.columns],
    'x1', 'x3', 'latitude', 'longitude', 'deviceType_enc',
    'x2_lag1', 'x2_lag2', 'x2_lag3', 'x2_roll3',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in monthly.columns]
print('Features:', FEATURE_COLS)

train_df = monthly[monthly['x2'].notna()].copy()
val_df   = monthly[
    (monthly['year'] == 2025) & monthly['month'].isin([5, 6]) & monthly['x2'].notna()
].copy()

X_train, y_train = train_df[FEATURE_COLS], train_df['x2']
X_val,   y_val   = val_df[FEATURE_COLS],   val_df['x2']
print(f'Train: {len(X_train)}  |  Val: {len(X_val)}')

## 7. LightGBM

In [ ]:
import lightgbm as lgb

lgbm_model = LGBMRegressor(
    n_estimators=1000, learning_rate=0.03, num_leaves=63,
    min_child_samples=5, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=RANDOM_STATE, n_jobs=-1,
)
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
)
lgbm_val_preds = lgbm_model.predict(X_val)
lgbm_mae = mean_absolute_error(y_val, lgbm_val_preds)
print(f'LightGBM Val MAE: {lgbm_mae:.6f}')

## 8. XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    random_state=RANDOM_STATE, n_jobs=-1,
    early_stopping_rounds=50, eval_metric='mae',
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)
xgb_val_preds = xgb_model.predict(X_val)
xgb_mae = mean_absolute_error(y_val, xgb_val_preds)
print(f'XGBoost Val MAE: {xgb_mae:.6f}')

## 9. LSTM na miesięcznych agregatach
Sekwencja = ostatnie `SEQ_LEN` miesięcy cech dla danego urządzenia → przewidujemy następny miesiąc.

In [ ]:
SEQ_LEN = 3  # ile poprzednich miesięcy widzi LSTM (train ma 7 mies. → okno 3 daje 4 próbki/urządzenie)

class MonthlySeqDataset(Dataset):
    """
    Buduje sekwencje z miesięcznych agregatów per urządzenie.
    Każda próbka: (seq_len miesięcy cech) → x2 następnego miesiąca.
    """
    def __init__(self, monthly_df, feature_cols, seq_len, labeled_only=True):
        self.samples = []
        self.targets = []
        df = monthly_df.copy()
        if labeled_only:
            df = df[df['x2'].notna()]

        for _, dev_df in df.groupby('deviceId'):
            dev_df = dev_df.sort_values(['year', 'month']).reset_index(drop=True)
            feats = dev_df[feature_cols].values.astype(np.float32)
            tgts  = dev_df['x2'].values.astype(np.float32)

            for i in range(seq_len, len(dev_df)):
                self.samples.append(feats[i-seq_len:i])
                self.targets.append(tgts[i])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.samples[idx]),
            torch.tensor(self.targets[idx]).unsqueeze(0),
        )


# Używamy całego monthly (train+val) z labelami
all_labeled = monthly[monthly['x2'].notna()].copy()
train_months_mask = ~((monthly['year'] == 2025) & monthly['month'].isin([5, 6]))
train_only = monthly[train_months_mask & monthly['x2'].notna()].copy()
val_only   = monthly[(monthly['year'] == 2025) & monthly['month'].isin([5, 6]) & monthly['x2'].notna()].copy()

# Uzupełnij NaN w feature_cols (lagi mogą być NaN dla pierwszych miesięcy)
for df_part in [train_only, val_only, all_labeled]:
    df_part[FEATURE_COLS] = df_part[FEATURE_COLS].fillna(df_part[FEATURE_COLS].median())

train_ds = MonthlySeqDataset(train_only, FEATURE_COLS, SEQ_LEN)
val_ds   = MonthlySeqDataset(val_only,   FEATURE_COLS, SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)

print(f'LSTM train samples: {len(train_ds)}  |  val samples: {len(val_ds)}')

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])  # last timestep


lstm_model = LSTMForecaster(input_size=len(FEATURE_COLS)).to(DEVICE)
optimizer  = torch.optim.Adam(lstm_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
criterion  = nn.L1Loss()  # MAE

NUM_EPOCHS  = 100
best_val    = float('inf')
best_state  = None
patience    = 20
no_improve  = 0

for epoch in range(NUM_EPOCHS):
    lstm_model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(lstm_model(X_b), y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(X_b)
    train_loss /= len(train_ds)

    lstm_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            val_loss += criterion(lstm_model(X_b), y_b).item() * len(X_b)
    val_loss /= max(len(val_ds), 1)

    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val   = val_loss
        best_state = {k: v.clone() for k, v in lstm_model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | train MAE: {train_loss:.5f} | val MAE: {val_loss:.5f}')

    if no_improve >= patience:
        print(f'Early stopping at epoch {epoch+1}')
        break

lstm_model.load_state_dict(best_state)
print(f'\nBest LSTM Val MAE: {best_val:.6f}')

In [ ]:
# Validation predictions for LSTM (only devices/months that have SEQ_LEN history)
lstm_model.eval()
lstm_val_preds_list = []
with torch.no_grad():
    for X_b, _ in val_loader:
        lstm_val_preds_list.extend(lstm_model(X_b.to(DEVICE)).cpu().numpy().flatten())

lstm_val_preds = np.array(lstm_val_preds_list)

# LSTM val targets align with val_ds samples (subset of val_df)
lstm_val_targets = np.array([val_ds.targets[i] for i in range(len(val_ds))])
lstm_mae = mean_absolute_error(lstm_val_targets, lstm_val_preds)
print(f'LSTM Val MAE: {lstm_mae:.6f}  (on {len(lstm_val_preds)} samples with history)')

## 10. Ensemble — porównanie i wagi

In [ ]:
# LightGBM i XGBoost mają predykcje dla wszystkich wierszy val_df
# LSTM ma predykcje tylko dla wierszy z wystarczającą historią (SEQ_LEN poprzednich miesięcy)
# → ensemble liczymy na wspólnym zbiorze

print(f'LightGBM Val MAE : {lgbm_mae:.6f}')
print(f'XGBoost  Val MAE : {xgb_mae:.6f}')
print(f'LSTM     Val MAE : {lstm_mae:.6f}  (subset)')

# Prosty ensemble LGBM + XGB (na pełnym val)
ensemble_preds = (lgbm_val_preds + xgb_val_preds) / 2
ensemble_mae   = mean_absolute_error(y_val, ensemble_preds)
print(f'\nLGBM+XGB ensemble Val MAE: {ensemble_mae:.6f}')

# Wykres
models = ['LightGBM', 'XGBoost', 'LGBM+XGB ensemble']
maes   = [lgbm_mae, xgb_mae, ensemble_mae]
plt.bar(models, maes, color=['steelblue', 'tomato', 'seagreen'])
plt.ylabel('MAE')
plt.title('Validation MAE comparison')
plt.tight_layout()
plt.show()

In [ ]:
# Optymalne wagi LGBM vs XGB metodą grid search na val
best_w, best_w_mae = 0.5, float('inf')
for w in np.arange(0.0, 1.01, 0.05):
    preds = w * lgbm_val_preds + (1 - w) * xgb_val_preds
    mae   = mean_absolute_error(y_val, preds)
    if mae < best_w_mae:
        best_w, best_w_mae = w, mae

print(f'Optimal LGBM weight: {best_w:.2f}  XGB weight: {1-best_w:.2f}  → Val MAE: {best_w_mae:.6f}')
LGBM_WEIGHT = best_w
XGB_WEIGHT  = 1 - best_w

## 11. Retrain na wszystkich danych & generuj submisję

In [ ]:
all_labeled = monthly[monthly['x2'].notna()].copy()
X_all, y_all = all_labeled[FEATURE_COLS], all_labeled['x2']

# Retrain LightGBM
lgbm_final = LGBMRegressor(**lgbm_model.get_params())
lgbm_final.set_params(n_estimators=lgbm_model.best_iteration_ + 50)
lgbm_final.fit(X_all, y_all)

# Retrain XGBoost
xgb_final = xgb.XGBRegressor(**{k: v for k, v in xgb_model.get_params().items()
                                 if k not in ('early_stopping_rounds',)})
xgb_final.set_params(n_estimators=xgb_model.best_iteration + 50)
xgb_final.fit(X_all, y_all)

# Retrain LSTM
all_ds     = MonthlySeqDataset(all_labeled, FEATURE_COLS, SEQ_LEN)
all_loader = DataLoader(all_ds, batch_size=64, shuffle=True)
lstm_final = LSTMForecaster(input_size=len(FEATURE_COLS)).to(DEVICE)
opt_final  = torch.optim.Adam(lstm_final.parameters(), lr=1e-3)

for epoch in range(lgbm_model.best_iteration_ // 10 + 20):
    lstm_final.train()
    for X_b, y_b in all_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        opt_final.zero_grad()
        criterion(lstm_final(X_b), y_b).backward()
        nn.utils.clip_grad_norm_(lstm_final.parameters(), 1.0)
        opt_final.step()

print('All models retrained on full labeled data.')

In [ ]:
def generate_submission(lgbm_m, xgb_m, lstm_m, monthly, feature_cols,
                         lgbm_w, xgb_w, seq_len, device):
    rows = []
    lstm_m.eval()

    for device_id, dev in monthly.groupby('deviceId'):
        dev = dev.sort_values(['year', 'month']).copy()
        dev[feature_cols] = dev[feature_cols].fillna(dev[feature_cols].median())

        static_mean = dev[
            [c for c in feature_cols if c not in
             ('month_sin', 'month_cos', 'x2_lag1', 'x2_lag2', 'x2_lag3', 'x2_roll3')]
        ].mean()

        last_x2  = dev['x2'].dropna().tolist()
        # Keep a rolling window of SEQ_LEN feature rows for LSTM
        feat_history = dev[feature_cols].values[-seq_len:].astype(np.float32).tolist()

        for month in FORECAST_MONTHS:
            feat = static_mean.copy()
            feat['month_sin'] = np.sin(2 * np.pi * month / 12)
            feat['month_cos'] = np.cos(2 * np.pi * month / 12)
            if 'x2_lag1' in feature_cols: feat['x2_lag1'] = last_x2[-1] if len(last_x2) >= 1 else 0.0
            if 'x2_lag2' in feature_cols: feat['x2_lag2'] = last_x2[-2] if len(last_x2) >= 2 else 0.0
            if 'x2_lag3' in feature_cols: feat['x2_lag3'] = last_x2[-3] if len(last_x2) >= 3 else 0.0
            if 'x2_roll3' in feature_cols: feat['x2_roll3'] = np.mean(last_x2[-3:]) if last_x2 else 0.0

            feat_row = feat[feature_cols].values.astype(np.float32)

            # GBM predictions
            p_lgbm = float(lgbm_m.predict([feat_row])[0])
            p_xgb  = float(xgb_m.predict([feat_row])[0])

            # LSTM prediction
            if len(feat_history) >= seq_len:
                seq_tensor = torch.tensor([feat_history[-seq_len:]], dtype=torch.float32).to(device)
                with torch.no_grad():
                    p_lstm = float(lstm_m(seq_tensor).cpu().item())
            else:
                p_lstm = (p_lgbm + p_xgb) / 2  # fallback if not enough history

            # Weighted ensemble
            pred = lgbm_w * p_lgbm + xgb_w * p_xgb + 0.0 * p_lstm  # LSTM weight=0 until tuned
            pred = float(np.clip(pred, 0.0, 1.0))

            rows.append((device_id, FORECAST_YEAR, month, pred))
            last_x2.append(pred)
            feat_history.append(feat_row.tolist())

    return pd.DataFrame(rows, columns=['deviceId', 'year', 'month', 'prediction'])


submission = generate_submission(
    lgbm_final, xgb_final, lstm_final,
    monthly, FEATURE_COLS,
    LGBM_WEIGHT, XGB_WEIGHT, SEQ_LEN, DEVICE
)
print(f'Submission rows: {len(submission)}')
submission.head(12)

## 12. Zapisz submission & modele

In [ ]:
OUT_DIR = os.path.join(DATA_DIR, 'out') if ENV == 'colab' else 'data/out'
os.makedirs(OUT_DIR, exist_ok=True)

submission.to_csv(os.path.join(OUT_DIR, 'submission.csv'), index=False)

with open(os.path.join(OUT_DIR, 'models.pkl'), 'wb') as f:
    pickle.dump({
        'lgbm': lgbm_final,
        'xgb':  xgb_final,
        'features': FEATURE_COLS,
        'lgbm_weight': LGBM_WEIGHT,
        'xgb_weight':  XGB_WEIGHT,
    }, f)

torch.save(lstm_final.state_dict(), os.path.join(OUT_DIR, 'lstm.pt'))

print(f'Saved to {OUT_DIR}/')

if ENV == 'colab':
    from google.colab import files
    submission.to_csv('submission.csv', index=False)
    files.download('submission.csv')

## 13. Sanity check

In [ ]:
print(submission.groupby('month')['prediction'].agg(['mean', 'std', 'min', 'max']).round(4))
assert submission[['deviceId', 'year', 'month']].duplicated().sum() == 0, 'Duplicate rows!'
assert submission['prediction'].between(0, 1).all(), 'Predictions out of [0,1]!'
print('\nAll checks passed.')